In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!git clone https://github.com/HarshAggarwal524/hinglish-sentiment-analysis
%cd /kaggle/working/hinglish-sentiment-analysis


fatal: destination path 'hinglish-sentiment-analysis' already exists and is not an empty directory.
/kaggle/working/hinglish-sentiment-analysis


In [7]:
!ls hinglish-sentiment-analysis/data/


ls: cannot access 'hinglish-sentiment-analysis/data/': No such file or directory


In [4]:
!pip install transformers datasets accelerate evaluate -q

In [5]:
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

sys.path.append("/kaggle/working/hinglish-sentiment-analysis")
from src.data_loader import load_conll

In [8]:
train_df = load_conll("data/hinglish_train.txt")
dev_df = load_conll("data/hinglish_dev.txt")
print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)

print("\nTrain sentiment distribution:")
print(train_df['sentiment'].value_counts())

print("\nDev sentiment distribution:")
print(dev_df['sentiment'].value_counts())

print("\nTrain sample:")
print(train_df.head(3))

Train shape: (15131, 3)
Dev shape: (3000, 3)

Train sentiment distribution:
sentiment
neutral     5638
positive    5034
negative    4459
Name: count, dtype: int64

Dev sentiment distribution:
sentiment
neutral     1128
positive     982
negative     890
Name: count, dtype: int64

Train sample:
  tweet_id sentiment                                               text
0        3  negative  @ AdilNisarButt pakistan ka ghra tauq he Pakis...
1       41  negative  Madarchod mulle ye mathura me Nahi dikha tha j...
2       48  positive  @ narendramodi Manya Pradhan Mantri mahoday Sh...


In [9]:
# TWEET COUNTS
print("=" * 50)
print("TWEET COUNTS (meta lines)")
print("=" * 50)
!grep -c "^meta" hinglish-sentiment-analysis/data/hinglish_train.txt
!grep -c "^meta" hinglish-sentiment-analysis/data/hinglish_dev.txt
!grep -c "^meta" hinglish-sentiment-analysis/data/hinglish_test.txt
!grep -c "^meta" hinglish-sentiment-analysis/data/hinglish_trial.txt

# GENERATE ID FILES
!grep "^meta" hinglish-sentiment-analysis/data/hinglish_train.txt | awk '{print $2}' | sort > /tmp/train_ids.txt
!grep "^meta" hinglish-sentiment-analysis/data/hinglish_dev.txt | awk '{print $2}' | sort > /tmp/dev_ids.txt
!grep "^meta" hinglish-sentiment-analysis/data/hinglish_test.txt | awk '{print $2}' | sort > /tmp/test_ids.txt
!grep "^meta" hinglish-sentiment-analysis/data/hinglish_trial.txt | awk '{print $2}' | sort > /tmp/trial_ids.txt

# OVERLAP CHECKS
print("=" * 50)
print("OVERLAP CHECKS")
print("=" * 50)
print("Train ∩ Dev:")
!comm -12 /tmp/train_ids.txt /tmp/dev_ids.txt | wc -l
print("Train ∩ Test:")
!comm -12 /tmp/train_ids.txt /tmp/test_ids.txt | wc -l
print("Train ∩ Trial:")
!comm -12 /tmp/train_ids.txt /tmp/trial_ids.txt | wc -l
print("Dev ∩ Test:")
!comm -12 /tmp/dev_ids.txt /tmp/test_ids.txt | wc -l
print("Dev ∩ Trial:")
!comm -12 /tmp/dev_ids.txt /tmp/trial_ids.txt | wc -l
print("Test ∩ Trial:")
!comm -12 /tmp/test_ids.txt /tmp/trial_ids.txt | wc -l

# NULL AND DUPLICATE CHECKS
print("=" * 50)
print("NULL AND DUPLICATE CHECKS")
print("=" * 50)
print("Train nulls:", train_df.isnull().sum().sum())
print("Dev nulls:  ", dev_df.isnull().sum().sum())
print("Train duplicate IDs:", train_df['tweet_id'].duplicated().sum())
print("Dev duplicate IDs:  ", dev_df['tweet_id'].duplicated().sum())
print("Train unique sentiments:", train_df['sentiment'].unique())
print("Dev unique sentiments:  ", dev_df['sentiment'].unique())

TWEET COUNTS (meta lines)
grep: hinglish-sentiment-analysis/data/hinglish_train.txt: No such file or directory
grep: hinglish-sentiment-analysis/data/hinglish_dev.txt: No such file or directory
grep: hinglish-sentiment-analysis/data/hinglish_test.txt: No such file or directory
grep: hinglish-sentiment-analysis/data/hinglish_trial.txt: No such file or directory
grep: hinglish-sentiment-analysis/data/hinglish_train.txt: No such file or directory
grep: hinglish-sentiment-analysis/data/hinglish_dev.txt: No such file or directory
grep: hinglish-sentiment-analysis/data/hinglish_test.txt: No such file or directory
grep: hinglish-sentiment-analysis/data/hinglish_trial.txt: No such file or directory
OVERLAP CHECKS
Train ∩ Dev:
0
Train ∩ Test:
0
Train ∩ Trial:
0
Dev ∩ Test:
0
Dev ∩ Trial:
0
Test ∩ Trial:
0
NULL AND DUPLICATE CHECKS
Train nulls: 0
Dev nulls:   0
Train duplicate IDs: 0
Dev duplicate IDs:   0
Train unique sentiments: ['negative' 'positive' 'neutral']
Dev unique sentiments:   ['posi

In [10]:

from sklearn.model_selection import train_test_split

# Combine and deduplicate
full_df = pd.concat([train_df, dev_df]).drop_duplicates(subset='tweet_id').reset_index(drop=True)
print("Total unique tweets:", len(full_df))

# 85/15 stratified split
train_custom, dev_custom = train_test_split(
    full_df,
    test_size=0.15,
    stratify=full_df['sentiment'],
    random_state=42
)

train_custom = train_custom.reset_index(drop=True)
dev_custom = dev_custom.reset_index(drop=True)

print("Train:", len(train_custom))
print("Dev:", len(dev_custom))

# Verify zero overlap
overlap = set(train_custom['tweet_id']) & set(dev_custom['tweet_id'])
print("Overlap:", len(overlap))  # must be 0

# Verify sentiment distribution
print("\nTrain distribution:\n", train_custom['sentiment'].value_counts())
print("\nDev distribution:\n", dev_custom['sentiment'].value_counts())


Total unique tweets: 15480
Train: 13158
Dev: 2322
Overlap: 0

Train distribution:
 sentiment
neutral     4919
positive    4368
negative    3871
Name: count, dtype: int64

Dev distribution:
 sentiment
neutral     868
positive    771
negative    683
Name: count, dtype: int64


In [12]:
train_custom.to_csv("data/train_custom.csv", index=False)
dev_custom.to_csv("data/dev_custom.csv", index=False)

print("Saved train_custom.csv:", len(train_custom), "tweets")
print("Saved dev_custom.csv:", len(dev_custom), "tweets")

Saved train_custom.csv: 13158 tweets
Saved dev_custom.csv: 2322 tweets


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train = vectorizer.fit_transform(train_custom['text'])
X_dev = vectorizer.transform(dev_custom['text'])

y_train = train_custom['sentiment']
y_dev = dev_custom['sentiment']

# Train LR
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

# Evaluate
preds = lr.predict(X_dev)
print("Accuracy:", accuracy_score(y_dev, preds))
print("Macro F1:", f1_score(y_dev, preds, average='macro'))
print("\nClassification Report:")
print(classification_report(y_dev, preds))

Accuracy: 0.5874246339362619
Macro F1: 0.592466908635028

Classification Report:
              precision    recall  f1-score   support

    negative       0.63      0.63      0.63       683
     neutral       0.51      0.51      0.51       868
    positive       0.64      0.63      0.64       771

    accuracy                           0.59      2322
   macro avg       0.59      0.59      0.59      2322
weighted avg       0.59      0.59      0.59      2322



In [14]:
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

train_custom["label"] = train_custom["sentiment"].map(label2id)
dev_custom["label"] = dev_custom["sentiment"].map(label2id)

print(train_custom[['sentiment', 'label']].head(5))

  sentiment  label
0  positive      2
1  negative      0
2  negative      0
3  negative      0
4  negative      0


In [15]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = Dataset.from_pandas(train_custom[["text", "label"]]).map(tokenize_function, batched=True)
dev_ds = Dataset.from_pandas(dev_custom[["text", "label"]]).map(tokenize_function, batched=True)

print("Train size:", len(train_ds))
print("Dev size:", len(dev_ds))

Map:   0%|          | 0/13158 [00:00<?, ? examples/s]

Map:   0%|          | 0/2322 [00:00<?, ? examples/s]

Train size: 13158
Dev size: 2322


In [16]:
# Rename label to labels
train_ds = train_ds.rename_column("label", "labels")
dev_ds = dev_ds.rename_column("label", "labels")

In [21]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)

MODEL_NAME = "bert-base-multilingual-cased"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="./results_mbert_custom",

    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=1e-5,
    weight_decay=0.05,
    warmup_steps=200,
    label_smoothing_factor=0.05,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

eval_results = trainer.evaluate()
print("\nEvaluation Results:")
print(eval_results)

pred_output = trainer.predict(dev_ds)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["negative", "neutral", "positive"],
        digits=4,
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.816862,1.825616,0.574935,0.580501
2,1.704654,1.761882,0.603359,0.607690
3,1.557923,1.797293,0.603790,0.606990


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:668] . unexpected pos 1001963392 vs 1001963284

In [22]:
!rm -rf ./results_mbert_custom

In [24]:
import os
import gc
import shutil
import inspect
import numpy as np
import torch

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

# --------------------------------------------------
# Clean previous checkpoints and GPU memory
# --------------------------------------------------

OUTPUT_DIR = "./results_mbert_custom"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

set_seed(42)

# --------------------------------------------------
# Load fresh pretrained mBERT
# --------------------------------------------------

MODEL_NAME = "bert-base-multilingual-cased"

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

# --------------------------------------------------
# Metrics
# --------------------------------------------------

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }

# --------------------------------------------------
# Training arguments
# --------------------------------------------------

args_dict = dict(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,
    label_smoothing_factor=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=1,
    report_to="none",

    seed=42,
    data_seed=42,
)

# Add only if your transformers version supports it.
# This avoids saving huge optimizer.pt files.
if "save_only_model" in inspect.signature(TrainingArguments.__init__).parameters:
    args_dict["save_only_model"] = True

training_args = TrainingArguments(**args_dict)

# --------------------------------------------------
# Trainer
# --------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics,
)

# --------------------------------------------------
# Train from scratch, not from old checkpoint
# --------------------------------------------------

train_result = trainer.train(resume_from_checkpoint=False)

# --------------------------------------------------
# Final evaluation
# --------------------------------------------------

eval_results = trainer.evaluate()

print("\nFinal Evaluation Results:")
print(eval_results)

# --------------------------------------------------
# Classification report and confusion matrix
# --------------------------------------------------

pred_output = trainer.predict(dev_ds)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["negative", "neutral", "positive"],
        digits=4,
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.862684,1.864743,0.570198,0.574169
2,1.716397,1.833983,0.604220,0.608077
3,1.528932,1.875434,0.596038,0.600575
4,1.382465,2.021469,0.600345,0.601100
5,1.260421,2.096365,0.605512,0.607854


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Final Evaluation Results:
{'eval_loss': 1.8344823122024536, 'eval_accuracy': 0.6037898363479759, 'eval_f1': 0.60762884671168, 'eval_runtime': 11.4626, 'eval_samples_per_second': 202.572, 'eval_steps_per_second': 3.228, 'epoch': 5.0}

Classification Report:
              precision    recall  f1-score   support

    negative     0.5946    0.6808    0.6348       683
     neutral     0.5435    0.5323    0.5378       868
    positive     0.6884    0.6161    0.6502       771

    accuracy                         0.6038      2322
   macro avg     0.6089    0.6097    0.6076      2322
weighted avg     0.6067    0.6038    0.6037      2322


Confusion Matrix:
[[465 184  34]
 [225 462 181]
 [ 92 204 475]]
